In [2]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Data Extraction from NASA

import requests
import json

API_KEY = "GO2zcuSMmfbZAmtPsF7tu7Emp6ohbnLRRJYYb2k2"

url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-01&end_date=2024-01-07&api_key={API_KEY}"

# Storing all asteroid records here
asteroid_data = []

while len(asteroid_data) < 10000:
    print(f" Fetching: {url}")
    response = requests.get(url)

    if response.status_code != 200:
        print(f" API Error: {response.status_code}")
        break

    data = response.json()
    neo_by_date = data.get("near_earth_objects", {})

    for date in neo_by_date:
        for asteroid in neo_by_date[date]:
            for approach in asteroid.get("close_approach_data", []):
                asteroid_data.append({
                    "id": int(asteroid.get("id", 0)),
                    "neo_reference_id": int(asteroid.get("neo_reference_id", 0)),
                    "name": asteroid.get("name", "Unknown"),
                    "absolute_magnitude_h": float(asteroid.get("absolute_magnitude_h", 0.0)),
                    "estimated_diameter_min_km": float(asteroid.get("estimated_diameter", {}).get("kilometers", {}).get("estimated_diameter_min", 0.0)),
                    "estimated_diameter_max_km": float(asteroid.get("estimated_diameter", {}).get("kilometers", {}).get("estimated_diameter_max", 0.0)),
                    "is_potentially_hazardous_asteroid": bool(asteroid.get("is_potentially_hazardous_asteroid", False)),
                    "close_approach_date": approach.get("close_approach_date"),
                    "relative_velocity_kmph": float(approach.get("relative_velocity", {}).get("kilometers_per_hour", 0.0)),
                    "astronomical": float(approach.get("miss_distance", {}).get("astronomical", 0.0)),
                    "miss_distance_km": float(approach.get("miss_distance", {}).get("kilometers", 0.0)),
                    "miss_distance_lunar": float(approach.get("miss_distance", {}).get("lunar", 0.0)),
                    "orbiting_body": approach.get("orbiting_body", "Unknown")
                })

    print(f" Records Collected so far: {len(asteroid_data)}")

    # Get the 'next' URL from the response
    next_url = data.get("links", {}).get("next", None)
    if not next_url:
        print(" No more pages available.")
        break

    url = next_url

    # Function to STOP after 10000 records collected
    if len(asteroid_data) >= 10000:
        break

with open("nasa_neo_asteroid_data.json", "w") as f:
    json.dump(asteroid_data, f, indent=2)

print(f" Total number of Records Collected {len(asteroid_data)}")

 Fetching: https://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-01&end_date=2024-01-07&api_key=GO2zcuSMmfbZAmtPsF7tu7Emp6ohbnLRRJYYb2k2
 Records Collected so far: 111
 Fetching: http://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-07&end_date=2024-01-13&detailed=false&api_key=GO2zcuSMmfbZAmtPsF7tu7Emp6ohbnLRRJYYb2k2
 Records Collected so far: 243
 Fetching: http://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-13&end_date=2024-01-19&detailed=false&api_key=GO2zcuSMmfbZAmtPsF7tu7Emp6ohbnLRRJYYb2k2
 Records Collected so far: 372
 Fetching: http://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-19&end_date=2024-01-25&detailed=false&api_key=GO2zcuSMmfbZAmtPsF7tu7Emp6ohbnLRRJYYb2k2
 Records Collected so far: 494
 Fetching: http://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-25&end_date=2024-01-31&detailed=false&api_key=GO2zcuSMmfbZAmtPsF7tu7Emp6ohbnLRRJYYb2k2
 Records Collected so far: 631
 Fetching: http://api.nasa.gov/neo/rest/v1/feed?start_date=2024-01-31&end_date=2024-02-06

In [9]:
# 🧹 Data Cleaning Steps

import json
from datetime import datetime

# Load your previously saved JSON data
with open("nasa_neo_asteroid_data.json", "r") as f:
    raw_data = json.load(f)

cleaned_asteroids = []
cleaned_approaches = []

seen_ids = set()

for record in raw_data:
    try:
        # Extraction and validation fields
        asteroid_id = int(record.get("id", 0))
        name = record.get("name", "Unknown")
        abs_mag = float(record.get("absolute_magnitude_h", 0.0))
        dia_min = float(record.get("estimated_diameter_min_km", 0.0))
        dia_max = float(record.get("estimated_diameter_max_km", 0.0))
        hazardous = bool(record.get("is_potentially_hazardous_asteroid", False))

        approach_date_str = record.get("close_approach_date")
        if not approach_date_str:
            continue
        approach_date = datetime.strptime(approach_date_str, "%Y-%m-%d").date()

        velocity = float(record.get("relative_velocity_kmph", 0.0))
        astro_dist = float(record.get("astronomical", 0.0))
        miss_km = float(record.get("miss_distance_km", 0.0))
        miss_lunar = float(record.get("miss_distance_lunar", 0.0))
        orbiting = record.get("orbiting_body", "Unknown")

        if asteroid_id not in seen_ids:
            cleaned_asteroids.append({
                "id": asteroid_id,
                "name": name,
                "absolute_magnitude_h": abs_mag,
                "estimated_diameter_min_km": dia_min,
                "estimated_diameter_max_km": dia_max,
                "is_potentially_hazardous_asteroid": hazardous
            })
            seen_ids.add(asteroid_id)

        cleaned_approaches.append({
            "neo_reference_id": asteroid_id,
            "close_approach_date": approach_date.isoformat(),  # convert to string for SQL compatibility
            "relative_velocity_kmph": velocity,
            "astronomical": astro_dist,
            "miss_distance_km": miss_km,
            "miss_distance_lunar": miss_lunar,
            "orbiting_body": orbiting
        })

    except Exception as e:
        print(f" Skipped record due to error: {e}")
        continue

with open("cleaned_asteroids_data.json", "w") as f:
    json.dump(cleaned_asteroids, f, indent=2)

with open("cleaned_close_approaches_data.json", "w") as f:
    json.dump(cleaned_approaches, f, indent=2)

print(f" Cleaned Asteroids: {len(cleaned_asteroids)}")
print(f" Cleaned Approaches: {len(cleaned_approaches)}")

 Cleaned Asteroids: 7404
 Cleaned Approaches: 10052


In [11]:
import sqlite3
import json

# Loading cleaned data
with open("cleaned_asteroids_data.json", "r") as f:
    asteroids = json.load(f)

with open("cleaned_close_approaches_data.json", "r") as f:
    approaches = json.load(f)

# Create or connect to SQLite DB
conn = sqlite3.connect("nasa_neo.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS asteroids (
    id INT,
    name TEXT,
    absolute_magnitude_h FLOAT,
    estimated_diameter_min_km FLOAT,
    estimated_diameter_max_km FLOAT,
    is_potentially_hazardous_asteroid BOOLEAN
)
""")

# Close_Approach Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS close_approach (
    neo_reference_id INT,
    close_approach_date DATE,
    relative_velocity_kmph FLOAT,
    astronomical FLOAT,
    miss_distance_km FLOAT,
    miss_distance_lunar FLOAT,
    orbiting_body TEXT
)
""")

# Inserting into Asteroid's Table
for a in asteroids:
    cursor.execute("""
    INSERT INTO asteroids (
        id, name, absolute_magnitude_h,
        estimated_diameter_min_km, estimated_diameter_max_km,
        is_potentially_hazardous_asteroid
    ) VALUES (?, ?, ?, ?, ?, ?)
    """, (
        a["id"],
        a["name"],
        a["absolute_magnitude_h"],
        a["estimated_diameter_min_km"],
        a["estimated_diameter_max_km"],
        a["is_potentially_hazardous_asteroid"]
    ))

for ca in approaches:
    cursor.execute("""
    INSERT INTO close_approach (
        neo_reference_id, close_approach_date,
        relative_velocity_kmph, astronomical,
        miss_distance_km, miss_distance_lunar, orbiting_body
    ) VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        ca["neo_reference_id"],
        ca["close_approach_date"],
        ca["relative_velocity_kmph"],
        ca["astronomical"],
        ca["miss_distance_km"],
        ca["miss_distance_lunar"],
        ca["orbiting_body"]
    ))

# Commit and close
conn.commit()
conn.close()

print(" Data inserted into 'nasa_neo.db' with tables 'asteroids' and 'close_approach'")

 Data inserted into 'nasa_neo.db' with tables 'asteroids' and 'close_approach'


In [13]:
import sqlite3

conn = sqlite3.connect("nasa_neo.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
conn.close()

print("Available tables:", tables)

Available tables: [('asteroids',), ('close_approach',)]
